# Uplift Modeling for Optimizing E-Commerce Advertising Campaigns

Dataset: [Social Media Advertisement Performance](https://www.kaggle.com/datasets/alperenmyung/social-media-advertisement-performance)

This notebook implements the uplift-modeling workflow described in the accompanying research report, covering data preparation, multiple meta-learners, causal forests, and interpretability with SHAP.

## Project Roadmap

1. Load and audit the raw Kaggle dataset.
2. Build a user-level analytical dataset ready for uplift modeling.
3. Train T-, S-, and X-learners plus a causal forest to estimate heterogeneous treatment effects.
4. Evaluate models with Qini curves and AUUC metrics.
5. Interpret the leading model with SHAP values and incremental gains tables.

### Working Assumptions

- Treatment corresponds to ad exposure/impressions; control rows are labeled accordingly in `ad_events`.
- Outcome corresponds to purchase/conversion events.
- Update the flags in the preprocessing section if the dataset uses different column names.

## Environment Setup

Install the required libraries once per environment. Uncomment the following cell if you need to add dependencies or the Kaggle API.

In [ ]:
# Install dependencies as needed
# !pip install -q kaggle scikit-uplift shap econml lightgbm


### Download and Cache the Dataset

Authenticate with Kaggle (place `kaggle.json` in `~/.kaggle/`) and download the dataset into a local `data/` directory.

In [ ]:
from pathlib import Path
import os

DATA_DIR = Path("data/social_media_ads")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download once using the Kaggle API (uncomment to execute)
# from kaggle.api.kaggle_api_extended import KaggleApi
# api = KaggleApi()
# api.authenticate()
# api.dataset_download_files(
#     "alperenmyung/social-media-advertisement-performance",
#     path=DATA_DIR,
#     unzip=True,
# )

raw_files = list(DATA_DIR.glob("*.csv"))
if not raw_files:
    raise FileNotFoundError(
        "CSV files not found. Please download and unzip the Kaggle dataset into 'data/social_media_ads/'."
    )


## Data Loading and Initial Exploration

In [ ]:
import pandas as pd
import numpy as np


def load_csv(name: str, parse_dates=None) -> pd.DataFrame:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Expected file '{name}' in {DATA_DIR.resolve()}.")
    return pd.read_csv(path, parse_dates=parse_dates)


users = load_csv("users.csv")
ad_events_sample = pd.read_csv(DATA_DIR / "ad_events.csv", nrows=5)
event_parse_cols = ["event_time"] if "event_time" in ad_events_sample.columns else None
ad_events = load_csv("ad_events.csv", parse_dates=event_parse_cols)
ads = load_csv("ads.csv")
campaigns = load_csv("campaigns.csv")

users.head()


In [ ]:
ad_events.head()


In [ ]:
ads.head()


In [ ]:
campaigns.head()


### Standardize Column Names

In [ ]:
for frame in (users, ad_events, ads, campaigns):
    frame.columns = frame.columns.str.strip().str.lower()

print("Users columns:", users.columns.tolist())
print("Ad events columns:", ad_events.columns.tolist())
print("Ads columns:", ads.columns.tolist())
print("Campaigns columns:", campaigns.columns.tolist())


### Event Type Distribution

In [ ]:
if "event_type" in ad_events.columns:
    ad_events["event_type"].value_counts(dropna=False)
else:
    raise KeyError("Expected an 'event_type' column in ad_events.csv.")


### Missing-Value Overview

In [ ]:
def describe_missing(df: pd.DataFrame, name: str) -> pd.DataFrame:
    missing = df.isna().mean().sort_values(ascending=False)
    return pd.DataFrame({"column": missing.index, f"{name}_missing_pct": (missing.values * 100).round(2)})


missing_frames = [
    describe_missing(users, "users"),
    describe_missing(ad_events, "ad_events"),
    describe_missing(ads, "ads"),
    describe_missing(campaigns, "campaigns"),
]

pd.concat(missing_frames, axis=0).reset_index(drop=True)


## Feature Engineering for Uplift Modeling

### Derive Treatment and Outcome Flags

The following cell infers treatment/control membership and conversion outcomes. Update the label sets if the dataset uses different names.

In [ ]:
TREATMENT_LABELS = {"impression", "exposure", "treatment", "shown"}
CONTROL_LABELS = {"control", "holdout"}
CONVERSION_LABELS = {"purchase", "conversion", "order"}

if "event_type" not in ad_events.columns:
    raise KeyError("The 'event_type' column was not found. Please adjust the label inference logic.")


event_types = ad_events["event_type"].astype(str).str.lower().str.strip()

ad_events["is_conversion"] = event_types.isin(CONVERSION_LABELS).astype(int)

if event_types.isin(CONTROL_LABELS).any():
    ad_events["is_treatment"] = (~event_types.isin(CONTROL_LABELS)).astype(int)
elif event_types.isin(TREATMENT_LABELS).any():
    ad_events["is_treatment"] = event_types.isin(TREATMENT_LABELS).astype(int)
elif "is_treatment" in ad_events.columns:
    ad_events["is_treatment"] = ad_events["is_treatment"].astype(int)
else:
    raise ValueError(
        "Unable to determine treatment indicator. Please map the dataset's treatment/control labels in TREATMENT_LABELS/CONTROL_LABELS or provide an 'is_treatment' column."
    )

ad_events[["event_type", "is_treatment", "is_conversion"]].head()


### Build a User-Level Analytical Dataset

In [ ]:
if "user_id" not in users.columns or "user_id" not in ad_events.columns:
    raise KeyError("The dataset must include 'user_id' in both users.csv and ad_events.csv.")


def pivot_counts(df: pd.DataFrame, index: str, column: str, prefix: str) -> pd.DataFrame:
    if column not in df.columns:
        return pd.DataFrame()
    pivot = (
        df.assign(_cnt=1)
        .pivot_table(index=index, columns=column, values="_cnt", aggfunc="sum", fill_value=0)
    )
    pivot.columns = [f"{prefix}{str(c)}" for c in pivot.columns]
    return pivot


events_augmented = ad_events.copy()
if "ad_id" in events_augmented.columns and "ad_id" in ads.columns:
    events_augmented = events_augmented.merge(ads, on="ad_id", how="left")
if "campaign_id" in events_augmented.columns and "campaign_id" in campaigns.columns:
    events_augmented = events_augmented.merge(campaigns, on="campaign_id", how="left")

user_conversion = ad_events.groupby("user_id")["is_conversion"].max().rename("conversion")
user_treatment = ad_events.groupby("user_id")["is_treatment"].max().rename("treatment")

event_type_pivot = pivot_counts(ad_events, "user_id", "event_type", "event_")
platform_pivot = pivot_counts(events_augmented, "user_id", "ad_platform", "platform_") if "ad_platform" in events_augmented.columns else pd.DataFrame()
adtype_pivot = pivot_counts(events_augmented, "user_id", "ad_type", "adtype_") if "ad_type" in events_augmented.columns else pd.DataFrame()

campaign_numeric_cols = [col for col in ["duration_days", "total_budget"] if col in events_augmented.columns]
if campaign_numeric_cols:
    campaign_agg = (
        events_augmented.groupby("user_id")[campaign_numeric_cols]
        .agg(['mean', 'max', 'sum'])
    )
    campaign_agg.columns = ["_".join(filter(None, map(str, col))).strip("_") for col in campaign_agg.columns]
else:
    campaign_agg = pd.DataFrame()

frames_to_join = [df for df in [event_type_pivot, platform_pivot, adtype_pivot, campaign_agg] if not df.empty]

user_features = users.set_index("user_id")
for df in frames_to_join:
    user_features = user_features.join(df, how="left")
user_features = user_features.join(user_treatment, how="inner")
user_features = user_features.join(user_conversion, how="left")

categorical_fill_values = {col: "Unknown" for col in user_features.select_dtypes(include="object").columns}
user_features = user_features.fillna(value=categorical_fill_values)
user_features["conversion"] = user_features["conversion"].fillna(0).astype(int)
user_features = user_features.fillna(0)

user_features.head()


### Prepare Training Matrices

In [ ]:
feature_df = user_features.reset_index()
target_col = "conversion"
treatment_col = "treatment"

feature_cols = [c for c in feature_df.columns if c not in {"user_id", target_col, treatment_col}]
categorical_cols = [c for c in feature_cols if feature_df[c].dtype == "object"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print(f"Total features: {len(feature_cols)} | Numeric: {len(numeric_cols)} | Categorical: {len(categorical_cols)}")


In [ ]:
from sklearn.model_selection import train_test_split

X = feature_df[feature_cols]
y = feature_df[target_col].astype(int)
treatment = feature_df[treatment_col].astype(int)

X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    X, y, treatment, test_size=0.3, stratify=treatment, random_state=42
)

X_train.shape, X_test.shape


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse=True), categorical_cols),
    ],
    remainder="drop",
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)


def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else matrix


X_train_array = to_dense(X_train_enc)
X_test_array = to_dense(X_test_enc)

feature_names = preprocessor.get_feature_names_out()
feature_names[:10]


## Meta-Learner Implementations

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from typing import Optional


class TLearner:
    def __init__(self, base_estimator=None):
        self.model_t = clone(base_estimator or GradientBoostingClassifier(random_state=42))
        self.model_c = clone(base_estimator or GradientBoostingClassifier(random_state=42))

    def fit(self, X, treatment, y):
        self.model_t.fit(X[treatment == 1], y[treatment == 1])
        self.model_c.fit(X[treatment == 0], y[treatment == 0])
        return self

    def predict_uplift(self, X):
        proba_t = self.model_t.predict_proba(X)[:, 1]
        proba_c = self.model_c.predict_proba(X)[:, 1]
        return proba_t - proba_c


class SLearner:
    def __init__(self, base_estimator=None):
        self.model = clone(base_estimator or GradientBoostingClassifier(random_state=42))

    def fit(self, X, treatment, y):
        augmented = np.column_stack([X, treatment])
        self.model.fit(augmented, y)
        return self

    def predict_uplift(self, X):
        treated = np.column_stack([X, np.ones(X.shape[0])])
        control = np.column_stack([X, np.zeros(X.shape[0])])
        proba_t = self.model.predict_proba(treated)[:, 1]
        proba_c = self.model.predict_proba(control)[:, 1]
        return proba_t - proba_c


class XLearner:
    def __init__(
        self,
        outcome_model=None,
        effect_model=None,
        propensity: Optional[float] = None,
    ):
        self.mu0 = clone(outcome_model or GradientBoostingClassifier(random_state=42))
        self.mu1 = clone(outcome_model or GradientBoostingClassifier(random_state=42))
        self.tau0 = clone(effect_model or GradientBoostingRegressor(random_state=42))
        self.tau1 = clone(effect_model or GradientBoostingRegressor(random_state=42))
        self.propensity = propensity

    def fit(self, X, treatment, y):
        treated_idx = treatment == 1
        control_idx = treatment == 0

        self.mu1.fit(X[treated_idx], y[treated_idx])
        self.mu0.fit(X[control_idx], y[control_idx])

        d1 = y[treated_idx] - self.mu0.predict_proba(X[treated_idx])[:, 1]
        d0 = self.mu1.predict_proba(X[control_idx])[:, 1] - y[control_idx]

        self.tau1.fit(X[treated_idx], d1)
        self.tau0.fit(X[control_idx], d0)

        self.propensity = self.propensity or float(treatment.mean())
        return self

    def predict_uplift(self, X):
        tau1_pred = self.tau1.predict(X)
        tau0_pred = self.tau0.predict(X)
        p = self.propensity
        return (1 - p) * tau1_pred + p * tau0_pred


### Causal Forest Estimator

In [ ]:
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

causal_forest = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=42),
    model_t=RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42),
    n_estimators=500,
    min_samples_leaf=20,
    random_state=42,
)

causal_forest.fit(y_train.values, t_train.values, X=X_train_array)

cf_uplift_test = causal_forest.effect(X_test_array)
cf_uplift_test[:5]


## Model Training & Evaluation

In [ ]:
from sklift.metrics import qini_auc_score, uplift_auc_score, uplift_at_k
from collections import OrderedDict

learners = OrderedDict(
    {
        "T-Learner": TLearner(),
        "S-Learner": SLearner(),
        "X-Learner": XLearner(),
    }
)

uplift_scores = {}
results = []

for name, learner in learners.items():
    learner.fit(X_train_array, t_train.values, y_train.values)
    uplift_pred = learner.predict_uplift(X_test_array)
    uplift_scores[name] = uplift_pred
    qini = qini_auc_score(y_test.values, uplift_pred, t_test.values)
    auuc = uplift_auc_score(y_test.values, uplift_pred, t_test.values)
    top10 = uplift_at_k(y_test.values, uplift_pred, t_test.values, strategy="overall", k=0.1)
    results.append({"model": name, "qini_auc": qini, "auuc": auuc, "uplift_at_10pct": top10})

cf_qini = qini_auc_score(y_test.values, cf_uplift_test, t_test.values)
cf_auuc = uplift_auc_score(y_test.values, cf_uplift_test, t_test.values)
cf_top10 = uplift_at_k(y_test.values, cf_uplift_test, t_test.values, strategy="overall", k=0.1)
results.append({"model": "Causal Forest", "qini_auc": cf_qini, "auuc": cf_auuc, "uplift_at_10pct": cf_top10})
uplift_scores["Causal Forest"] = cf_uplift_test

result_df = pd.DataFrame(results).sort_values(by="qini_auc", ascending=False).reset_index(drop=True)
result_df


### Qini and Uplift Curves

In [ ]:
import matplotlib.pyplot as plt
from sklift.viz import plot_qini_curve, plot_uplift_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, scores in uplift_scores.items():
    plot_qini_curve(y_test.values, scores, t_test.values, ax=axes[0], label=name)
    plot_uplift_curve(y_test.values, scores, t_test.values, ax=axes[1], label=name)

axes[0].set_title("Qini Curves")
axes[1].set_title("Uplift Curves")
for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()


### Incremental Gains Table

In [ ]:
from sklift.metrics import uplift_by_percentile

best_model_name = result_df.loc[0, "model"]
best_scores = uplift_scores[best_model_name]

gains_table = uplift_by_percentile(
    y_test.values,
    best_scores,
    t_test.values,
    strategy="overall",
    total=True,
    n_bins=10,
)

gains_table


## SHAP Interpretability

In [ ]:
import shap

best_model_name = result_df.loc[0, "model"]
print(f"Best uplift model: {best_model_name}")

X_train_frame = pd.DataFrame(X_train_array, columns=feature_names)
X_test_frame = pd.DataFrame(X_test_array, columns=feature_names)

if best_model_name == "T-Learner":
    shap_model = learners["T-Learner"].model_t
elif best_model_name == "S-Learner":
    shap_model = learners["S-Learner"].model
elif best_model_name == "X-Learner":
    shap_model = learners["X-Learner"].tau1
else:
    shap_model = None

if shap_model is None:
    print("SHAP plots are skipped for the causal forest. Use 'causal_forest.feature_importances_' for a global view.")
else:
    background = shap.sample(X_train_frame, min(200, len(X_train_frame)), random_state=42)
    explainer = shap.TreeExplainer(shap_model, data=background)
    shap_values = explainer.shap_values(X_test_frame)
    if isinstance(shap_values, list) and len(shap_values) > 1:
        shap_values = shap_values[1]
    shap.summary_plot(shap_values, X_test_frame, plot_type="bar", max_display=15)


In [ ]:
if 'shap_values' in globals() and shap_model is not None:
    shap.summary_plot(shap_values, X_test_frame, max_display=15)


## Next Steps

- Calibrate hyperparameters of each learner via cross-validation with uplift-aware scoring.
- Track incremental ROI by combining predicted uplift with revenue or margin per conversion.
- Experiment with alternative evaluation metrics (e.g., pROCini) and fairness diagnostics.